# Alias-config reconstruction — PEDV

Đưa tool một **PEDV mới tinh** (chỉ tên gene của reference, chưa có alias map): tầng gợi ý dựa-trên-tọa-độ
dựng lại được config curated tới đâu?

**Ba bước, gọi ĐÚNG hàm pipeline** (`ui/stages/bootstrap_alias.py`), không reimplement:
`build_coordinate_supported_alias_suggestions` → `review_uncertain_alias_suggestions` (LLM) →
`apply_approved_alias_suggestions` (dựng `config_temp`).

> **Đo hai chiều, per-canonical:**
> - **PRECISION** (duyệt config_temp): tool bỏ tên gì dưới canon X → truth có đồng ý thuộc X không?
>   Bắt `wrong_gene` (map sai), `false_save` (lưu nhầm rác).
> - **RECALL** (duyệt config_truth, **gate theo corpus**): alias thật truth có dưới X **mà thực sự xuất
>   hiện trong 100 record** → config_temp có giữ ở X không? Bắt `missed` (kiểu bug sM: alias thật bị loại).
>
> **Gate corpus (quan trọng):** alias truth **không** xuất hiện trong 100 record thì tool không có gì để
> gợi ý → **park, không tính accuracy**. Nếu không gate sẽ phạt oan tool.
>
> **Bước người không tự động được:** harness dừng ở khuyến nghị + chính sách auto-approve khai báo rõ
> (chấp nhận mọi dòng tool đánh `save`). Số là **cận trên**. Không API key → LLM mock (in rõ chế độ).

## Setup

In [1]:
from pathlib import Path
import os, sys
import pandas as pd

ROOT = Path.cwd()
for c in [ROOT, *ROOT.parents]:
    if (c / "app" / "src").exists():
        ROOT = c; break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "app" / "validation" / "03_alias_suggestion"))

# Nạp .env (giống UI) để LLMConfig thấy OPENAI_API_KEY
env = ROOT / ".env"
if env.exists():
    for line in env.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("="); os.environ.setdefault(k.strip(), v.strip())

import importlib
import config_reconstruction as R
importlib.reload(R)
from app.src.llm.config import LLMConfig

VIRUS = "PEDV"
MIN_IOU = 0.90
PROVIDER = None if LLMConfig.from_env().available else R._MockLLMProvider()
print("LLM:", "REAL" if PROVIDER is None else "MOCK (no API key)")

LLM: REAL


## Chạy — seed → suggest → LLM → config_temp → so per-canonical

Ghi ra `outputs/config_temp_PEDV.json`, `reconstruction_PEDV.tsv` (per-canonical), `..._detail.tsv` (từng ca sai).

In [2]:
per_canon, detail = R.run_virus(VIRUS, R.DATASETS[VIRUS], MIN_IOU, llm_provider=PROVIDER)
R.summarize(VIRUS, per_canon)


===== PEDV =====
  PRECISION (config_temp → truth): 24/25 = 96.0%
     wrong_gene 1  false_save 0  not_in_truth 0
  RECALL (truth ∩ corpus → config_temp): 24/30 = 80.0%
     missed 6  (parked, not scored: 14)


## Bảng per-canonical (đọc từng gene)

In [ ]:
pd.set_option("display.width", 200)
per_canon

## Precision — chỗ tool lưu SAI

`wrong_gene` (map nhầm gene) và `false_save` (lưu nhầm rác) là lỗi đắt, lý tưởng = 0.
`not_in_truth` = tool gợi ý mà truth im lặng → **kiểm truth có thiếu không**.

In [4]:
prec_issues = detail[detail["side"]=="precision"]
prec_issues if len(prec_issues) else "— precision sạch, không lưu sai gì —"

,canonical,side,name,issue
4,ORF1a,precision,orf1ab,wrong_gene(ORF1ab)


## Recall — chỗ tool BỎ SÓT (đây là chỗ bắt bug kiểu sM)

`missed(temp→...)` cho biết alias thật bị đẩy đi đâu: `excluded` = bị loại nhầm (nguy hiểm nhất),
một canon khác = map lệch.

In [5]:
miss = detail[(detail["side"]=="recall") & (detail["issue"].str.startswith("missed"))]
miss if len(miss) else "— recall đầy đủ, không bỏ sót alias nào có trong corpus —"

,canonical,side,name,issue
0,E,recall,sm,missed(temp→None)
1,E,recall,smprotein,missed(temp→None)
2,E,recall,smallmembraneprotein,missed(temp→excluded)
3,N,recall,nucleoprotein,missed(temp→None)
19,ORF3,recall,mp,missed(temp→None)
20,ORF3,recall,accessorymembraneprotein,missed(temp→excluded)


## Parked — alias truth KHÔNG có trong corpus (không tính điểm)

Đây là alias curated từ literature/kinh nghiệm mà lô 100 record này không dùng. Tool không thể gợi ý được,
nên loại khỏi mẫu số recall. Liệt kê để minh bạch.

In [6]:
parked = detail[detail["issue"]=="parked_not_in_corpus"]
print(f"{len(parked)} alias park (không tính accuracy)")
parked[["canonical","name"]]

0 alias park (không tính accuracy)


,canonical,name


## Ghi chú cho paper

- **Precision** = độ tin của cái tool tự lưu; **Recall** (gate corpus) = độ phủ trên phần tool *có thể*
  gợi ý. Hai số tách riêng, không gộp.
- **Gate corpus** là bắt buộc để công bằng: alias literature-only không phải lỗi lifting.
- **Recall detail là nơi lộ bug kiểu sM** — alias thật có trong corpus mà bị `missed(temp→excluded)`.